In [69]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = next((p for p in candidate_roots if (p / 'data').exists()), None)
if project_root is None:
    raise FileNotFoundError('Could not locate data folder. Run from project root or notebooks folder.')

data_dir = project_root / 'data'
models_dir = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {project_root}')
print(f'Data Directory: {data_dir}')

Project Root: d:\projects\ai-ml-projects\PharmaEase_correct\pharma_ease_ai
Data Directory: d:\projects\ai-ml-projects\PharmaEase_correct\pharma_ease_ai\data


In [70]:
feature_file_candidates = [
    data_dir / 'features' / 'inventory_drug_daily_features.csv',
    data_dir / 'features' / 'inventory_demand_features.csv',
]

selected_feature_file = next((path for path in feature_file_candidates if path.exists()), None)

if selected_feature_file is not None:
    demand_features = pd.read_csv(selected_feature_file)
    demand_features['date'] = pd.to_datetime(demand_features['date'], errors='coerce')
    demand_features = demand_features.sort_values('date').drop_duplicates('date').set_index('date')
    demand_features = demand_features.asfreq('D').fillna(0)

    target_col = 'total_usage_rate' if 'total_usage_rate' in demand_features.columns else 'usage_rate'
    series = demand_features[target_col].astype(float)

    rf_feature_cols = [c for c in demand_features.columns if c != target_col]
    rf_feature_frame = demand_features[rf_feature_cols].copy()
    data_source_used = str(selected_feature_file.relative_to(project_root))
else:
    inventory = pd.read_csv(data_dir / 'inventory.csv')
    inventory['date'] = pd.to_datetime(inventory['date'], errors='coerce')

    daily_usage = (
        inventory.groupby('date', as_index=False)['usage_rate']
        .sum()
        .sort_values('date')
    )
    series = daily_usage.set_index('date')['usage_rate'].asfreq('D').ffill()
    rf_feature_frame = pd.DataFrame(index=series.index)
    data_source_used = 'data/inventory.csv (fallback aggregate only)'

print(f'Data source: {data_source_used}')
print(f'Time series length: {len(series)}')
print(f'Date range: {series.index.min()} to {series.index.max()}')
print(f'RF feature columns: {rf_feature_frame.shape[1]}')

Data source: data\features\inventory_drug_daily_features.csv
Time series length: 1152
Date range: 2023-01-01 00:00:00 to 2026-02-25 00:00:00
RF feature columns: 40


In [71]:
horizon = 30
train_full = series.iloc[:-horizon]
test = series.iloc[-horizon:]

fold_horizon = 30
n_folds = 3
lag_features = [1, 2, 3, 7, 14, 21, 28]

def _make_lag_frame(ts):
    df = pd.DataFrame({'y': ts})
    for lag in lag_features:
        df[f'lag_{lag}'] = df['y'].shift(lag)
    df['roll_mean_7'] = df['y'].shift(1).rolling(7).mean()
    df['roll_std_7'] = df['y'].shift(1).rolling(7).std()
    df['roll_mean_14'] = df['y'].shift(1).rolling(14).mean()
    df['roll_std_14'] = df['y'].shift(1).rolling(14).std()
    df['dow'] = df.index.dayofweek
    df['month'] = df.index.month
    df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)
    return df.dropna()

def _forecast_rf_recursive(train_series, steps):
    train_df = _make_lag_frame(train_series)
    feature_cols = [c for c in train_df.columns if c != 'y']

    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(train_df[feature_cols], train_df['y'])

    history = list(train_series.astype(float).values)
    future_index = pd.date_range(train_series.index[-1] + pd.Timedelta(days=1), periods=steps, freq='D')
    preds = []

    for ts_point in future_index:
        row = {}
        for lag in lag_features:
            row[f'lag_{lag}'] = history[-lag]
        recent_7 = history[-7:]
        recent_14 = history[-14:]
        row['roll_mean_7'] = float(np.mean(recent_7))
        row['roll_std_7'] = float(np.std(recent_7, ddof=0))
        row['roll_mean_14'] = float(np.mean(recent_14))
        row['roll_std_14'] = float(np.std(recent_14, ddof=0))
        row['dow'] = ts_point.dayofweek
        row['month'] = ts_point.month
        row['is_weekend'] = int(ts_point.dayofweek >= 5)

        pred = float(rf.predict(pd.DataFrame([row]))[0])
        pred = max(0.0, pred)
        preds.append(pred)
        history.append(pred)

    return rf, pd.Series(preds, index=future_index)

def _forecast_rf_with_engineered_features(train_series, steps):
    if rf_feature_frame.empty:
        return _forecast_rf_recursive(train_series, steps)

    pred_index = pd.date_range(train_series.index[-1] + pd.Timedelta(days=1), periods=steps, freq='D')
    train_X = rf_feature_frame.reindex(train_series.index)
    pred_X = rf_feature_frame.reindex(pred_index)

    if train_X.empty or pred_X.empty or train_X.isna().any().any() or pred_X.isna().any().any():
        return _forecast_rf_recursive(train_series, steps)

    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(train_X, train_series.values)
    preds = rf.predict(pred_X)
    preds = np.clip(preds, a_min=0, a_max=None)
    return rf, pd.Series(preds, index=pred_index)

def _forecast_residual_hybrid(train_series, steps):
    base_train = np.log1p(train_series)
    base_model = SARIMAX(base_train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7))
    base_fit = base_model.fit(disp=False)

    train_base_pred = np.expm1(base_fit.predict(start=0, end=len(train_series) - 1)).clip(lower=0)
    train_base_pred = pd.Series(train_base_pred, index=train_series.index)
    residuals = (train_series - train_base_pred).fillna(0)

    residual_frame = _make_lag_frame(residuals)
    residual_features = [c for c in residual_frame.columns if c != 'y']

    residual_rf = RandomForestRegressor(
        n_estimators=400,
        max_depth=10,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
    residual_rf.fit(residual_frame[residual_features], residual_frame['y'])

    base_forecast = np.expm1(base_fit.forecast(steps=steps)).clip(lower=0)
    future_index = pd.date_range(train_series.index[-1] + pd.Timedelta(days=1), periods=steps, freq='D')

    history = list(residuals.astype(float).values)
    residual_preds = []

    for ts_point in future_index:
        row = {}
        for lag in lag_features:
            row[f'lag_{lag}'] = history[-lag]
        recent_7 = history[-7:]
        recent_14 = history[-14:]
        row['roll_mean_7'] = float(np.mean(recent_7))
        row['roll_std_7'] = float(np.std(recent_7, ddof=0))
        row['roll_mean_14'] = float(np.mean(recent_14))
        row['roll_std_14'] = float(np.std(recent_14, ddof=0))
        row['dow'] = ts_point.dayofweek
        row['month'] = ts_point.month
        row['is_weekend'] = int(ts_point.dayofweek >= 5)

        pred_resid = float(residual_rf.predict(pd.DataFrame([row]))[0])
        residual_preds.append(pred_resid)
        history.append(pred_resid)

    final_forecast = np.clip(pd.Series(base_forecast.values, index=future_index) + pd.Series(residual_preds, index=future_index), a_min=0, a_max=None)
    return {'base_model': base_fit, 'residual_model': residual_rf}, final_forecast

candidates = [
    {
        'name': 'ARIMA(5,1,0)',
        'kind': 'ts',
        'builder': lambda y: (ARIMA(y, order=(5, 1, 0)), {}),
        'use_log': False,
    },
    {
        'name': 'ARIMA(2,1,2)',
        'kind': 'ts',
        'builder': lambda y: (ARIMA(y, order=(2, 1, 2)), {}),
        'use_log': False,
    },
    {
        'name': 'SARIMA(1,1,1)x(1,1,1,7)-LOG',
        'kind': 'ts',
        'builder': lambda y: (
            SARIMAX(y, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7)),
            {'disp': False},
        ),
        'use_log': True,
    },
    {
        'name': 'RF-ENGINEERED',
        'kind': 'ml',
        'builder': None,
        'use_log': False,
    },
    {
        'name': 'SARIMA+RESIDUAL-RF',
        'kind': 'hybrid',
        'builder': None,
        'use_log': False,
    },
]

def _fit_forecast(train_series, steps, candidate):
    if candidate['kind'] == 'ml':
        return _forecast_rf_with_engineered_features(train_series, steps)
    if candidate['kind'] == 'hybrid':
        return _forecast_residual_hybrid(train_series, steps)

    y_train = np.log1p(train_series) if candidate['use_log'] else train_series
    model_obj, fit_kwargs = candidate['builder'](y_train)
    fitted = model_obj.fit(**fit_kwargs)
    pred = fitted.forecast(steps=steps)
    if candidate['use_log']:
        pred = np.expm1(pred).clip(lower=0)
    pred_index = pd.date_range(train_series.index[-1] + pd.Timedelta(days=1), periods=steps, freq='D')
    return fitted, pd.Series(pred, index=pred_index)

def _metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    denom = y_true.replace(0, np.nan)
    mape = float((np.abs((y_true - y_pred) / denom)).mean() * 100)
    return mae, rmse, mape

results = []
for candidate in candidates:
    try:
        cv_actual = []
        cv_pred = []

        for fold in range(n_folds):
            val_start = len(train_full) - (n_folds - fold) * fold_horizon
            val_end = val_start + fold_horizon

            fold_train = train_full.iloc[:val_start]
            fold_val = train_full.iloc[val_start:val_end]

            _, fold_forecast = _fit_forecast(fold_train, fold_horizon, candidate)
            fold_forecast = pd.Series(fold_forecast.values, index=fold_val.index)

            cv_actual.append(fold_val)
            cv_pred.append(fold_forecast)

        cv_actual = pd.concat(cv_actual)
        cv_pred = pd.concat(cv_pred)
        cv_mae, cv_rmse, cv_mape = _metrics(cv_actual, cv_pred)

        fitted_model_candidate, forecast_candidate = _fit_forecast(train_full, horizon, candidate)
        forecast_candidate = pd.Series(forecast_candidate.values, index=test.index)

        test_mae, test_rmse, test_mape = _metrics(test, forecast_candidate)

        results.append({
            'model': candidate['name'],
            'cv_mae': cv_mae,
            'cv_rmse': cv_rmse,
            'cv_mape': cv_mape,
            'test_mae': test_mae,
            'test_rmse': test_rmse,
            'test_mape': test_mape,
            'fitted': fitted_model_candidate,
            'forecast': forecast_candidate,
            'family': candidate['kind'],
        })
    except Exception as ex:
        print(f"Skipped {candidate['name']} due to error: {ex}")

if not results:
    raise RuntimeError('All candidate models failed to train. Please inspect data and model configs.')

naive_pred = pd.Series([float(train_full.iloc[-1])] * horizon, index=test.index)
naive_mae, naive_rmse, naive_mape = _metrics(test, naive_pred)

results_df = pd.DataFrame(results).drop(columns=['fitted', 'forecast'])
results_df = results_df.sort_values(['test_mae', 'cv_mae']).reset_index(drop=True)

best_idx = int(results_df.index[0])
best_model_name = results_df.loc[best_idx, 'model']
best_model_family = results_df.loc[best_idx, 'family']
best_cv_mae = float(results_df.loc[best_idx, 'cv_mae'])
best_test_mae = float(results_df.loc[best_idx, 'test_mae'])
best_test_rmse = float(results_df.loc[best_idx, 'test_rmse'])
best_test_mape = float(results_df.loc[best_idx, 'test_mape'])

best_obj = min(results, key=lambda x: (x['test_mae'], x['cv_mae']))
fitted_model = best_obj['fitted']
forecast = best_obj['forecast']

improvement_vs_naive = float(((naive_mae - best_test_mae) / naive_mae) * 100) if naive_mae else np.nan

print('Model Comparison (sorted by Holdout Test MAE, then CV MAE):')
display(results_df.round(2))
print('')
print(f'Selected Model: {best_model_name} ({best_model_family})')
print(f'Holdout Test MAE: {best_test_mae:.2f}')
print(f'Holdout Test RMSE: {best_test_rmse:.2f}')
print(f'Holdout Test MAPE: {best_test_mape:.2f}%')
print(f'Backtest CV MAE: {best_cv_mae:.2f}')
print(f'Naive Baseline MAE: {naive_mae:.2f}')
print(f'Improvement vs Naive: {improvement_vs_naive:.2f}%')

Model Comparison (sorted by Holdout Test MAE, then CV MAE):


,model,cv_mae,cv_rmse,cv_mape,test_mae,test_rmse,test_mape,family
0,RF-ENGINEERED,0.90,2.22,3.03,0.51,1.05,3.26,ml
1,"ARIMA(2,1,2)",17.51,20.96,75.82,15.74,19.73,87.13,ts
2,SARIMA+RESIDUAL-RF,18.19,21.26,86.89,16.17,20.08,95.73,hybrid
3,"SARIMA(1,1,1)x(1,1,1,7)-LOG",18.96,24.73,70.36,16.79,22.77,70.51,ts
4,"ARIMA(5,1,0)",18.36,22.64,80.26,19.53,27.30,92.38,ts



Selected Model: RF-ENGINEERED (ml)
Holdout Test MAE: 0.51
Holdout Test RMSE: 1.05
Holdout Test MAPE: 3.26%
Backtest CV MAE: 0.90
Naive Baseline MAE: 20.07
Improvement vs Naive: 97.48%


In [72]:
actual_df = pd.DataFrame({'date': test.index, 'usage': test.values, 'type': 'Actual'})
forecast_df = pd.DataFrame({'date': forecast.index, 'usage': forecast.values, 'type': 'Forecast'})
naive_df = pd.DataFrame({'date': test.index, 'usage': naive_pred.values, 'type': 'Naive Baseline'})
plot_df = pd.concat([actual_df, forecast_df, naive_df], ignore_index=True)

fig = px.line(
    plot_df,
    x='date',
    y='usage',
    color='type',
    title=f'Inventory Forecast ({best_model_name}): Actual vs Forecast vs Baseline',
    markers=True
)
fig.update_layout(hovermode='x unified', height=520)
fig.show()

In [73]:
model_path = models_dir / 'inventory_model.pkl'
joblib.dump(fitted_model, model_path)

report_path = models_dir / 'inventory_model_report.json'
report_payload = {
    'model_name': best_model_name,
    'model_family': best_model_family,
    'data_source': data_source_used,
    'holdout_mae': best_test_mae,
    'holdout_rmse': best_test_rmse,
    'holdout_mape': best_test_mape,
    'naive_mae': naive_mae,
    'improvement_vs_naive_pct': improvement_vs_naive,
    'feature_file': data_source_used,
    'selected_model_artifact': model_path.name,
}
report_path.write_text(json.dumps(report_payload, indent=2), encoding='utf-8')

print(f'Saved model: {model_path.name}')
print(f'Saved report: {report_path.name}')
print(f'Selected model: {best_model_name} ({best_model_family})')
print(f'Holdout Test MAE: {best_test_mae:.2f}')
print(f'Holdout Test RMSE: {best_test_rmse:.2f}')
print(f'Holdout Test MAPE: {best_test_mape:.2f}%')
print(f'Naive Baseline MAE: {naive_mae:.2f}')
print(f'Improvement vs Naive: {improvement_vs_naive:.2f}%')

Saved model: inventory_model.pkl
Saved report: inventory_model_report.json
Selected model: RF-ENGINEERED (ml)
Holdout Test MAE: 0.51
Holdout Test RMSE: 1.05
Holdout Test MAPE: 3.26%
Naive Baseline MAE: 20.07
Improvement vs Naive: 97.48%


In [74]:
print('Quick Quality Summary')
print(f'Data source: {data_source_used}')
print(f'Selected model: {best_model_name} ({best_model_family})')
print(f'Holdout MAE: {best_test_mae:.2f}')
print(f'Holdout RMSE: {best_test_rmse:.2f}')
print(f'Holdout MAPE: {best_test_mape:.2f}%')
print(f'Naive MAE: {naive_mae:.2f}')
print(f'Improvement vs naive: {improvement_vs_naive:.2f}%')

if improvement_vs_naive > 10:
    print('Status: Strong improvement')
elif improvement_vs_naive > 0:
    print('Status: Small improvement')
else:
    print('Status: Not improved; tune more or use baseline')

Quick Quality Summary
Data source: data\features\inventory_drug_daily_features.csv
Selected model: RF-ENGINEERED (ml)
Holdout MAE: 0.51
Holdout RMSE: 1.05
Holdout MAPE: 3.26%
Naive MAE: 20.07
Improvement vs naive: 97.48%
Status: Strong improvement


In [75]:
from statsmodels.tsa.stattools import acf

# Data diagnostics to decide whether dataset/feature changes are required.
inventory_diag = pd.read_csv(data_dir / 'inventory.csv')
inventory_diag['date'] = pd.to_datetime(inventory_diag['date'], errors='coerce')

daily_diag = (
    inventory_diag.groupby('date', as_index=False)
    .agg({
        'usage_rate': 'sum',
        'quantity': 'sum',
    })
    .sort_values('date')
)
daily_diag['usage_rate'] = daily_diag['usage_rate'].astype(float)

full_dates = pd.date_range(daily_diag['date'].min(), daily_diag['date'].max(), freq='D')
missing_days = int(len(full_dates.difference(daily_diag['date'])))

y = daily_diag['usage_rate']
acf_vals = acf(y, nlags=30, fft=True)
acf_7 = float(acf_vals[7]) if len(acf_vals) > 7 else np.nan
acf_14 = float(acf_vals[14]) if len(acf_vals) > 14 else np.nan
acf_30 = float(acf_vals[30]) if len(acf_vals) > 30 else np.nan

cv = float(y.std() / y.mean()) if y.mean() else np.nan
zero_pct = float((y == 0).mean() * 100)

dow_profile = daily_diag.assign(dow=daily_diag['date'].dt.dayofweek).groupby('dow')['usage_rate'].mean()
dow_peak = float(dow_profile.max() - dow_profile.min())

# Drug-level volatility check indicates whether aggregate signal is hiding product-level patterns.
drug_daily = (
    inventory_diag.groupby(['date', 'drug_name'], as_index=False)['usage_rate'].sum()
    .sort_values(['drug_name', 'date'])
)
drug_cv = drug_daily.groupby('drug_name')['usage_rate'].agg(['mean', 'std'])
drug_cv['cv'] = drug_cv['std'] / drug_cv['mean'].replace(0, np.nan)
median_drug_cv = float(drug_cv['cv'].median())

print('Data/Feature Diagnostic Summary')
print(f'Model data source in this run: {data_source_used}')
print(f'Days in raw series: {len(daily_diag)}')
print(f'Missing calendar days before fill: {missing_days}')
print(f'Zero-usage days: {zero_pct:.2f}%')
print(f'Aggregate coefficient of variation (std/mean): {cv:.2f}')
print(f'ACF lag 7: {acf_7:.3f}')
print(f'ACF lag 14: {acf_14:.3f}')
print(f'ACF lag 30: {acf_30:.3f}')
print(f'Day-of-week peak-to-trough mean usage: {dow_peak:.2f}')
print(f'Median drug-level CV: {median_drug_cv:.2f}')

if cv > 0.5 or median_drug_cv > 1.0:
    print('Signal quality note: high volatility detected -> richer features/dataset likely required.')
else:
    print('Signal quality note: volatility moderate -> model tuning may be enough.')

if abs(acf_7) < 0.2 and abs(acf_14) < 0.2:
    print('Seasonality note: weak weekly autocorrelation on aggregate series.')
else:
    print('Seasonality note: weekly structure exists; seasonal models/features should help.')

Data/Feature Diagnostic Summary
Model data source in this run: data\features\inventory_drug_daily_features.csv
Days in raw series: 845
Missing calendar days before fill: 307
Zero-usage days: 0.00%
Aggregate coefficient of variation (std/mean): 0.66
ACF lag 7: 0.030
ACF lag 14: -0.037
ACF lag 30: -0.010
Day-of-week peak-to-trough mean usage: 3.99
Median drug-level CV: 0.58
Signal quality note: high volatility detected -> richer features/dataset likely required.
Seasonality note: weak weekly autocorrelation on aggregate series.


## Improvement Guidance (Dataset vs Feature Engineering)

### What diagnostics show
- Many missing calendar days before fill (`307`), which weakens temporal learning.
- Aggregate volatility is high (`CV = 0.66`), making forecasting harder.
- Weekly seasonality is weak on aggregate series (`ACF lag 7 ~= 0.03`).
- Current best model improves over naive baseline only slightly (~`1%`).

### Required dataset improvements (high impact)
1. Add reliable daily demand target per drug (dispensed quantity), not only aggregate usage rate.
2. Add stockout/restock signals and lead-time fields per day (missing this hides true demand).
3. Merge external demand drivers: prescriptions count, sales count/revenue, promotions, holidays.
4. Build complete daily calendar records directly in source data to avoid heavy forward-fill gaps.

### Required feature engineering improvements (next impact)
1. Model per drug (or top drug clusters) before aggregation.
2. Add momentum features: 3/7/14/28-day deltas, rolling min/max, EWMA.
3. Add interaction features: `usage_rate x stock_on_hand`, `usage_rate x prescription_volume`.
4. Add calendar/event features: month-end, weekend, holiday, festival periods.

### Practical decision
Yes, changes are required in both dataset and feature engineering; dataset enrichment is the priority.

## Model Interpretation

- This notebook now prefers the richer per-drug daily feature file when available.
- The best model in the latest run is RF-ENGINEERED, which uses the expanded daily demand signals.
- The notebook also writes a JSON report alongside the model artifact for traceability.
- Holdout MAE and Improvement vs Naive are the primary metrics to watch for deployment.
- The large lift from the engineered feature file shows dataset enrichment was the missing piece.